# 🌍 World CNN System for Intelligent Image Recognition
## Complete Deep Learning Workflow: Classification · Optimization · Evaluation · Deployment

---
| Item | Detail |
|------|--------|
| **Dataset** | CIFAR-10 (60,000 images · 10 classes) |
| **Framework** | TensorFlow 2.x / Keras |
| **Workflow** | Data Prep → Preprocessing → Model → Eval → Improve → Augment → Transfer Learning → Tune → Deploy |
---


## ⚙️ Global Imports & Configuration

In [ ]:
import os, time, warnings, itertools, random
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image, ImageEnhance
from IPython.display import display

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, callbacks
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50, VGG16
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_score, recall_score, f1_score)

# ── Reproducibility ──────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# ── Style ─────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('husl')

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"NumPy      : {np.__version__}")
gpu = tf.config.list_physical_devices('GPU')
print(f"GPU        : {gpu if gpu else 'Not available — using CPU'}")

# ── Output directories ────────────────────────────────────
for d in ['outputs/plots','outputs/models','outputs/reports']:
    os.makedirs(d, exist_ok=True)
print("Output directories ready ✓")


---
# Section 1 — Dataset Preparation

### 1.1 Download & Explore CIFAR-10

In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
N_CLASSES = len(CLASS_NAMES)

print("╔══════════════════════════════════════════════╗")
print("║         CIFAR-10 DATASET SUMMARY             ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  Total images       : {len(X_train_raw)+len(X_test_raw):>6,}                  ║")
print(f"║  Training images    : {len(X_train_raw):>6,}                  ║")
print(f"║  Test images        : {len(X_test_raw):>6,}                  ║")
print(f"║  Number of classes  : {N_CLASSES:>6}                  ║")
print(f"║  Image dimensions   : {X_train_raw.shape[1:]}              ║")
print(f"║  Pixel dtype        : {X_train_raw.dtype}                ║")
print(f"║  Pixel range        : [{X_train_raw.min()}, {X_train_raw.max()}]               ║")
print("╚══════════════════════════════════════════════╝")


### 1.2 Dataset Summary Table

In [ ]:
import pandas as pd

summary_data = {
    'Split'           : ['Training', 'Validation', 'Test', 'Total'],
    'Images'          : [45000, 5000, 10000, 60000],
    'Percentage'      : ['75%', '8.3%', '16.7%', '100%'],
    'Purpose'         : [
        'Learn model weights via gradient descent',
        'Tune hyperparameters & detect overfitting',
        'Final unbiased performance evaluation',
        '—'
    ]
}
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))
print()
print("📌 Purpose of each split:")
print("  • Training   — The model sees and learns from these images (backprop).")
print("  • Validation — Monitored during training to guide hyperparameter choices.")
print("  • Test       — Held out until the very end; reflects real-world performance.")


### 1.3 Split Dataset (Train / Validation / Test)

In [ ]:
# Carve 5,000 samples from training set → validation
X_train_split = X_train_raw[:45000]
y_train_split = y_train_raw[:45000]
X_val         = X_train_raw[45000:]
y_val         = y_train_raw[45000:]
X_test_split  = X_test_raw
y_test_split  = y_test_raw

print(f"Training   : {X_train_split.shape}")
print(f"Validation : {X_val.shape}")
print(f"Test       : {X_test_split.shape}")


### 1.4 Visualize 12 Images from Different Classes

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.patch.set_facecolor('#1a1a2e')

# Pick 2 samples per class, show first 12
shown, seen_classes = 0, []
idx_map = {}
for i, label in enumerate(y_train_split.flatten()):
    if label not in seen_classes:
        seen_classes.append(label)
        idx_map[label] = i
    if len(seen_classes) == N_CLASSES:
        break

display_indices = [idx_map[c] for c in range(N_CLASSES)][:12]

for ax, idx in zip(axes.flat, display_indices):
    label = y_train_split[idx][0]
    ax.imshow(X_train_split[idx])
    ax.set_title(CLASS_NAMES[label], color='white', fontsize=12, fontweight='bold', pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('#4fc3f7'); spine.set_linewidth(2)
    ax.set_facecolor('#0d0d0d')

fig.suptitle('CIFAR-10 — One Sample per Class', color='white',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/plots/class_samples.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


### 1.5 Dataset Class Imbalance Analysis

In [ ]:
labels_flat = y_train_split.flatten()
class_counts = {CLASS_NAMES[i]: np.sum(labels_flat == i) for i in range(N_CLASSES)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = plt.cm.Set3(np.linspace(0, 1, N_CLASSES))
bars = ax1.bar(CLASS_NAMES, class_counts.values(), color=colors, edgecolor='black', linewidth=0.8)
ax1.axhline(np.mean(list(class_counts.values())), color='red', linestyle='--',
            linewidth=1.5, label=f'Mean = {np.mean(list(class_counts.values())):.0f}')
ax1.set_title('Samples per Class', fontweight='bold')
ax1.set_xlabel('Class'); ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=35)
ax1.legend()
for bar, cnt in zip(bars, class_counts.values()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+40,
             str(cnt), ha='center', va='bottom', fontsize=9)

# Pie chart
ax2.pie(class_counts.values(), labels=CLASS_NAMES, colors=colors,
        autopct='%1.1f%%', startangle=140, pctdistance=0.85)
ax2.set_title('Class Distribution (%)', fontweight='bold')

plt.suptitle('CIFAR-10 Class Balance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Class counts:")
for name, cnt in class_counts.items():
    bar = '█' * (cnt // 500)
    print(f"  {name:>12} : {cnt:>5}  {bar}")

imbalance = max(class_counts.values()) / min(class_counts.values())
print(f"\nImbalance ratio (max/min): {imbalance:.2f}x  →  {'Balanced ✓' if imbalance < 1.5 else 'Imbalanced ⚠'}")


---
# Section 2 — Image Preprocessing

### 2.1 Resize All Images to Uniform Dimensions

In [ ]:
# CIFAR-10 is already 32×32; we also prepare a 96×96 version for Transfer Learning
TARGET_SIZE_SMALL = (32, 32)   # for custom CNN
TARGET_SIZE_LARGE = (96, 96)   # for pretrained models

def resize_dataset(X, size):
    return np.array([cv2.resize(img, size) for img in X])

print(f"Standard size    : {TARGET_SIZE_SMALL} — used for custom CNN")
print(f"Large size        : {TARGET_SIZE_LARGE} — used for transfer learning")
print(f"Original shape   : {X_train_split.shape}")
# 32×32 is already correct; resize is demonstrated below
sample_resized = cv2.resize(X_train_split[0], (64, 64))
print(f"Resized sample   : {sample_resized.shape} (example at 64×64)")


### 2.2 Normalize Pixel Values to [0, 1]

In [ ]:
X_train = X_train_split.astype('float32') / 255.0
X_val   = X_val.astype('float32')         / 255.0
X_test  = X_test_split.astype('float32')  / 255.0

y_train = to_categorical(y_train_split, N_CLASSES)
y_val   = to_categorical(y_val,         N_CLASSES)
y_test  = to_categorical(y_test_split,  N_CLASSES)

print("Normalization complete.")
print(f"  Pixel range before : [0, 255]")
print(f"  Pixel range after  : [{X_train.min():.1f}, {X_train.max():.1f}]")
print()
print("📌 Why normalize?")
print("  1. Gradient stability — prevents vanishing/exploding gradients")
print("  2. Faster convergence — all features on the same scale")
print("  3. Weight initialization — assumes ~zero-mean, unit-variance inputs")
print("  4. Numerical precision — avoids overflow in activations like sigmoid")


### 2.3 Image Preprocessing Techniques

In [ ]:
def apply_preprocessing(img_uint8):
    """Apply noise removal, contrast & brightness correction."""
    # 1. Noise removal (Gaussian blur)
    denoised = cv2.GaussianBlur(img_uint8, (3, 3), 0)

    # 2. Contrast adjustment (CLAHE on L channel of LAB)
    lab  = cv2.cvtColor(denoised, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    contrasted = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # 3. Brightness correction (gamma = 1.2)
    gamma = 1.2
    lut = np.array([((i/255.0)**gamma)*255 for i in range(256)], dtype=np.uint8)
    brightened = cv2.LUT(contrasted, lut)

    return denoised, contrasted, brightened

sample = X_train_split[25]
denoised, contrasted, brightened = apply_preprocessing(sample)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
steps = ['Original', 'Grayscale', 'Denoised', 'Contrast Enhanced',
         'Brightness Corrected', 'Normalized', 'Edge Map', 'Final Array']

imgs = [
    sample,
    cv2.cvtColor(sample, cv2.COLOR_RGB2GRAY),
    denoised,
    contrasted,
    brightened,
    (brightened.astype('float32')/255.0),
    cv2.Canny(cv2.cvtColor(sample, cv2.COLOR_RGB2GRAY), 50, 150),
    (brightened.astype('float32')/255.0)
]
cmaps = [None,'gray',None,None,None,None,'gray',None]

for ax, img, title, cmap in zip(axes.flat, imgs, steps, cmaps):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Image Preprocessing Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/preprocessing_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.4 Convert to NumPy Arrays for CNN

In [ ]:
print("Final NumPy arrays ready for CNN training:")
print(f"  X_train : {X_train.shape}  dtype={X_train.dtype}")
print(f"  y_train : {y_train.shape}  (one-hot encoded)")
print(f"  X_val   : {X_val.shape}")
print(f"  y_val   : {y_val.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_test  : {y_test.shape}")
print(f"  Memory  : {(X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1e6:.1f} MB total")


---
# Section 3 — CNN Model Design

### 3.1 Architecture Diagram (ASCII)

In [ ]:
arch_diagram = """
┌─────────────────────────────────────────────────────────────┐
│                  INPUT  (32 × 32 × 3)                       │
├─────────────────────────────────────────────────────────────┤
│  BLOCK 1                                                     │
│   Conv2D  32 filters 3×3  ReLU  → (32×32×32)               │
│   BatchNorm                                                  │
│   Conv2D  32 filters 3×3  ReLU  → (32×32×32)               │
│   MaxPool 2×2              → (16×16×32)                     │
│   Dropout 0.25                                               │
├─────────────────────────────────────────────────────────────┤
│  BLOCK 2                                                     │
│   Conv2D  64 filters 3×3  ReLU  → (16×16×64)               │
│   BatchNorm                                                  │
│   Conv2D  64 filters 3×3  ReLU  → (16×16×64)               │
│   MaxPool 2×2              → (8×8×64)                       │
│   Dropout 0.25                                               │
├─────────────────────────────────────────────────────────────┤
│  BLOCK 3                                                     │
│   Conv2D  128 filters 3×3 ReLU  → (8×8×128)                │
│   BatchNorm                                                  │
│   Conv2D  128 filters 3×3 ReLU  → (8×8×128)                │
│   MaxPool 2×2              → (4×4×128)                      │
│   Dropout 0.25                                               │
├─────────────────────────────────────────────────────────────┤
│  CLASSIFIER HEAD                                             │
│   GlobalAveragePooling2D   → (128,)                         │
│   Dense 256  ReLU                                           │
│   Dropout 0.5                                               │
│   Dense 10   Softmax       → PREDICTIONS                    │
└─────────────────────────────────────────────────────────────┘
"""
print(arch_diagram)


### 3.2 Parameter Count per Layer

In [ ]:
param_table = [
    ("Conv2D 3×3×3→32",    "(3×3×3+1)×32",       896),
    ("Conv2D 3×3×32→32",   "(3×3×32+1)×32",     9_248),
    ("Conv2D 3×3×32→64",   "(3×3×32+1)×64",    18_496),
    ("Conv2D 3×3×64→64",   "(3×3×64+1)×64",    36_928),
    ("Conv2D 3×3×64→128",  "(3×3×64+1)×128",   73_856),
    ("Conv2D 3×3×128→128", "(3×3×128+1)×128", 147_584),
    ("Dense 128→256",      "(128+1)×256",       33_024),
    ("Dense 256→10",       "(256+1)×10",         2_570),
]
total = sum(p for _,_,p in param_table)

print(f"{'Layer':<28} {'Formula':<22} {'Params':>10}")
print("─"*62)
for layer, formula, params in param_table:
    print(f"{layer:<28} {formula:<22} {params:>10,}")
print("─"*62)
print(f"{'TOTAL PARAMETERS':<28} {'':22} {total:>10,}")
print(f"{'Approx. memory (float32)':<28} {'':22} {total*4/1e6:>9.1f}MB")


### 3.3 Implement & Display Model

In [ ]:
def build_world_cnn(input_shape=(32,32,3), n_classes=10, name='WorldCNN'):
    inp = keras.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    return keras.Model(inputs=inp, outputs=out, name=name)

model = build_world_cnn()
model.summary()


### 3.4 Train Model & Record Curves

In [ ]:
EPOCHS     = 30
BATCH_SIZE = 64

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb_list = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=7,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=4, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint('outputs/models/best_worldcnn.keras',
                              monitor='val_accuracy', save_best_only=True, verbose=0)
]

print("Training WorldCNN...")
t0 = time.time()
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=cb_list,
    verbose=1
)
print(f"Training time: {(time.time()-t0)/60:.1f} min")

# ── Plot training curves ─────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history.history['accuracy'])+1)

ax1.plot(ep, history.history['accuracy'],     lw=2, label='Train Acc', color='#2196F3')
ax1.plot(ep, history.history['val_accuracy'], lw=2, label='Val Acc',   color='#FF5722', ls='--')
ax1.fill_between(ep, history.history['accuracy'], history.history['val_accuracy'], alpha=0.1, color='gray')
ax1.set_title('Accuracy Curves', fontweight='bold'); ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ep, history.history['loss'],     lw=2, label='Train Loss', color='#4CAF50')
ax2.plot(ep, history.history['val_loss'], lw=2, label='Val Loss',   color='#F44336', ls='--')
ax2.set_title('Loss Curves', fontweight='bold'); ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('WorldCNN — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/training_history.png', dpi=150, bbox_inches='tight')
plt.show()


---
# Section 4 — Model Evaluation

### 4.1 Test Set Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
val_loss,  val_acc  = model.evaluate(X_val,  y_val,  verbose=0)

print("╔════════════════════════════════════╗")
print("║    MODEL EVALUATION RESULTS        ║")
print("╠════════════════════════════════════╣")
print(f"║  Validation Accuracy : {val_acc*100:>6.2f}%     ║")
print(f"║  Validation Loss     : {val_loss:>8.4f}     ║")
print(f"║  Test Accuracy       : {test_acc*100:>6.2f}%     ║")
print(f"║  Test Loss           : {test_loss:>8.4f}     ║")
print("╚════════════════════════════════════╝")


### 4.2 Confusion Matrix

In [ ]:
y_pred_probs = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='white')
ax1.set_title('Confusion Matrix (Counts)', fontweight='bold')
ax1.set_xlabel('Predicted Label'); ax1.set_ylabel('True Label')
ax1.tick_params(axis='x', rotation=40)

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax2,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='white', vmin=0, vmax=1)
ax2.set_title('Confusion Matrix (Normalized)', fontweight='bold')
ax2.set_xlabel('Predicted Label'); ax2.set_ylabel('True Label')
ax2.tick_params(axis='x', rotation=40)

plt.suptitle('WorldCNN — Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.3 Precision, Recall, F1-Score

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
df_report = pd.DataFrame(report).transpose().round(3)
print("Classification Report:")
print(df_report.to_string())

# Per-class bar chart
classes = CLASS_NAMES
prec   = [report[c]['precision'] for c in classes]
rec    = [report[c]['recall']    for c in classes]
f1     = [report[c]['f1-score']  for c in classes]

x = np.arange(len(classes))
w = 0.28

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w,   prec, w, label='Precision', color='#2196F3', edgecolor='black', lw=0.5)
ax.bar(x,       rec,  w, label='Recall',    color='#4CAF50', edgecolor='black', lw=0.5)
ax.bar(x + w,   f1,   w, label='F1-Score',  color='#FF9800', edgecolor='black', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(classes, rotation=35)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score'); ax.legend()
ax.axhline(0.9, color='red', ls='--', lw=1, alpha=0.5, label='0.90 threshold')
ax.set_title('Per-Class Precision / Recall / F1', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/plots/classification_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

macro_f1 = report['macro avg']['f1-score']
print(f"\nMacro F1-Score: {macro_f1:.4f}")


### 4.4 Visualize 10 Test Predictions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.patch.set_facecolor('#0d0d0d')

sample_idx = np.random.choice(len(X_test), 10, replace=False)
for ax, idx in zip(axes.flat, sample_idx):
    true_lbl = y_true[idx]
    pred_lbl = y_pred[idx]
    conf     = y_pred_probs[idx][pred_lbl]
    correct  = (true_lbl == pred_lbl)

    ax.imshow(X_test[idx])
    color = '#00e676' if correct else '#ff1744'
    icon  = '✓' if correct else '✗'
    ax.set_title(f"{icon} True: {CLASS_NAMES[true_lbl]}\nPred: {CLASS_NAMES[pred_lbl]} ({conf:.0%})",
                 color=color, fontsize=9, fontweight='bold')
    ax.axis('off')
    for sp in ax.spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(2.5)

plt.suptitle('Model Predictions on Test Set  (green=correct, red=wrong)',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/test_predictions.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d0d')
plt.show()


---
# Section 5 — Model Improvement

### 5.1 Deeper CNN (More Conv Layers)

In [ ]:
def build_deeper_cnn():
    inp = keras.Input(shape=(32,32,3))
    x = layers.Conv2D(32,  (3,3), padding='same', activation='relu')(inp)
    x = layers.Conv2D(32,  (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64,  (3,3), padding='same', activation='relu')(x)
    x = layers.Conv2D(64,  (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.2)(x)

    # Extra block
    x = layers.Conv2D(256, (3,3), padding='same', activation='relu')(x)
    x = layers.Conv2D(256, (3,3), padding='same', activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x); x = layers.Dropout(0.5)(x)
    out = layers.Dense(N_CLASSES, activation='softmax')(x)
    return keras.Model(inp, out, name='DeeperCNN')

deeper = build_deeper_cnn()
deeper.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Deeper CNN params: {deeper.count_params():,}")
h_deeper = deeper.fit(X_train, y_train, epochs=20, batch_size=64,
                      validation_data=(X_val, y_val),
                      callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
                      verbose=0)
_, acc_deeper = deeper.evaluate(X_test, y_test, verbose=0)
_, acc_base   = model.evaluate(X_test, y_test, verbose=0)
print(f"Base CNN accuracy : {acc_base:.4f}")
print(f"Deeper CNN accuracy: {acc_deeper:.4f}  Δ={acc_deeper-acc_base:+.4f}")


### 5.2 Dropout Analysis

In [ ]:
def build_with_dropout(drop_rate):
    inp = keras.Input(shape=(32,32,3))
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(drop_rate)(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(drop_rate)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(drop_rate)(x)
    out = layers.Dense(N_CLASSES, activation='softmax')(x)
    return keras.Model(inp, out)

dropout_rates = [0.0, 0.25, 0.5]
dropout_results = {}
for dr in dropout_rates:
    m = build_with_dropout(dr)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train, epochs=12, batch_size=64,
              validation_data=(X_val, y_val), verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    dropout_results[dr] = {'history': h, 'acc': acc}
    print(f"Dropout={dr:.2f}: test_acc={acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for dr, res in dropout_results.items():
    axes[0].plot(res['history'].history['accuracy'],     label=f'dr={dr} train', lw=1.5)
    axes[0].plot(res['history'].history['val_accuracy'], label=f'dr={dr} val',   lw=1.5, ls='--')
    axes[1].plot(res['history'].history['val_loss'],     label=f'dr={dr}', lw=2)
for ax in axes: ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[0].set_title('Dropout — Train vs Val Accuracy'); axes[1].set_title('Dropout — Val Loss')
plt.suptitle('Dropout Rate Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/dropout_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.3 Batch Normalization Impact

In [ ]:
def build_with_bn(use_bn):
    inp = keras.Input(shape=(32,32,3))
    x = layers.Conv2D(64, (3,3), padding='same')(inp)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), padding='same')(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    out = layers.Dense(N_CLASSES, activation='softmax')(x)
    return keras.Model(inp, out)

results_bn = {}
for use_bn in [False, True]:
    label = 'With BN' if use_bn else 'Without BN'
    m = build_with_bn(use_bn)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train, epochs=15, batch_size=64,
              validation_data=(X_val, y_val), verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    results_bn[label] = {'history': h, 'acc': acc}
    print(f"{label}: {acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, res in results_bn.items():
    ls = '-' if 'Without' in label else '--'
    axes[0].plot(res['history'].history['val_accuracy'], label=label, lw=2, ls=ls)
    axes[1].plot(res['history'].history['val_loss'],     label=label, lw=2, ls=ls)
for ax in axes: ax.legend(); ax.grid(True, alpha=0.3)
axes[0].set_title('Val Accuracy — Batch Norm Effect')
axes[1].set_title('Val Loss — Batch Norm Effect')
plt.suptitle('Batch Normalization: Training Stability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/batchnorm_impact.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.4 Activation Function Comparison

In [ ]:
activations = ['relu', 'elu', 'selu', 'tanh']
act_results  = {}

for act in activations:
    m = models.Sequential([
        keras.Input(shape=(32,32,3)),
        layers.Conv2D(64, (3,3), padding='same', activation=act),
        layers.MaxPooling2D(),
        layers.Conv2D(128,(3,3), padding='same', activation=act),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation=act),
        layers.Dense(N_CLASSES, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train, epochs=10, batch_size=64,
              validation_data=(X_val, y_val), verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    act_results[act] = {'history': h, 'acc': acc}
    print(f"{act:>6}: test_acc={acc:.4f}")

plt.figure(figsize=(10, 5))
for act, res in act_results.items():
    plt.plot(res['history'].history['val_accuracy'], label=act, lw=2)
plt.title('Activation Function Comparison — Val Accuracy', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/plots/activation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTest accuracy summary:")
for act, res in sorted(act_results.items(), key=lambda x: -x[1]['acc']):
    print(f"  {act:>6}: {res['acc']:.4f}")


---
# Section 6 — Data Augmentation

### 6.1 Configure ImageDataGenerator

In [ ]:
augment_gen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    fill_mode='nearest'
)

# Visualise augmented samples
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')

for row, cls_idx in enumerate([0, 1, 2, 3]):
    cls_samples = X_train[y_train.argmax(axis=1) == cls_idx][:1]
    axes[row][0].imshow(cls_samples[0]); axes[row][0].set_title(f'Orig: {CLASS_NAMES[cls_idx]}', fontsize=8)
    axes[row][0].axis('off')
    aug_iter = augment_gen.flow(cls_samples, batch_size=1)
    for col in range(1, 6):
        aug_img = next(aug_iter)[0]
        axes[row][col].imshow(np.clip(aug_img, 0, 1))
        axes[row][col].set_title(f'Aug {col}', fontsize=8)
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('outputs/plots/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()


### 6.2 Train with Augmented Data

In [ ]:
model_aug = build_world_cnn(name='WorldCNN_Augmented')
model_aug.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])

train_gen = augment_gen.flow(X_train, y_train, batch_size=64, seed=SEED)
steps     = len(X_train) // 64

h_aug = model_aug.fit(
    train_gen,
    steps_per_epoch=steps,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=[callbacks.EarlyStopping(patience=6, restore_best_weights=True)],
    verbose=0
)
_, acc_aug = model_aug.evaluate(X_test, y_test, verbose=0)
_, acc_orig = model.evaluate(X_test, y_test, verbose=0)
print(f"Without augmentation : {acc_orig:.4f}")
print(f"With augmentation    : {acc_aug:.4f}  Δ={acc_aug-acc_orig:+.4f}")


### 6.3 Augmentation Generalization Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep_orig = range(1, len(history.history['accuracy'])+1)
ep_aug  = range(1, len(h_aug.history['accuracy'])+1)

for ax, metric in zip(axes, ['accuracy', 'loss']):
    ax.plot(ep_orig, history.history[f'val_{metric}'], label='Original', lw=2, color='#2196F3')
    ax.plot(ep_aug,  h_aug.history[f'val_{metric}'],   label='Augmented', lw=2, color='#FF5722', ls='--')
    ax.set_title(f'Validation {metric.capitalize()}'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Original vs Augmented Data — Generalization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/augmentation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

gap_orig = max(history.history['accuracy']) - max(history.history['val_accuracy'])
gap_aug  = max(h_aug.history['accuracy'])   - max(h_aug.history['val_accuracy'])
print(f"Overfit gap original : {gap_orig:.4f}")
print(f"Overfit gap augmented: {gap_aug:.4f}")
print(f"Augmentation {'reduced ✓' if gap_aug < gap_orig else 'increased ⚠'} overfitting gap by {abs(gap_aug-gap_orig):.4f}")


---
# Section 7 — Transfer Learning

### 7.1 Load MobileNetV2 for Feature Extraction

In [ ]:
# Resize to 96×96 for MobileNetV2 (minimum 32 but 96 gives better features)
print("Resizing images for transfer learning (96×96)...")
X_train_96 = tf.image.resize(X_train, [96, 96]).numpy()
X_val_96   = tf.image.resize(X_val,   [96, 96]).numpy()
X_test_96  = tf.image.resize(X_test,  [96, 96]).numpy()
print(f"Resized: {X_train_96.shape}")

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mnv2_preprocess

base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
base.trainable = False

print(f"MobileNetV2: {len(base.layers)} layers, {base.count_params():,} params (frozen)")


### 7.2 Add Custom Classification Layers

In [ ]:
inp = keras.Input(shape=(96, 96, 3))
x = mnv2_preprocess(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(N_CLASSES, activation='softmax')(x)
tl_model = keras.Model(inp, out, name='MobileNetV2_Transfer')

tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Trainable params : {sum(v.numpy().size for v in tl_model.trainable_variables):,}")
print(f"Frozen params    : {sum(v.numpy().size for v in tl_model.non_trainable_variables):,}")

h_tl = tl_model.fit(X_train_96, y_train, epochs=12, batch_size=64,
                    validation_data=(X_val_96, y_val),
                    callbacks=[callbacks.EarlyStopping(patience=4, restore_best_weights=True)],
                    verbose=0)
_, acc_tl = tl_model.evaluate(X_test_96, y_test, verbose=0)
print(f"\nFeature Extraction Accuracy: {acc_tl:.4f}")


### 7.3 Fine-Tune Pretrained Model

In [ ]:
# Unfreeze last 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

ft_trainable = sum(1 for l in base.layers if l.trainable)
print(f"Fine-tuning {ft_trainable} layers in base.")

tl_model.compile(optimizer=keras.optimizers.Adam(1e-5),   # lower LR!
                 loss='categorical_crossentropy', metrics=['accuracy'])

h_ft = tl_model.fit(X_train_96, y_train, epochs=8, batch_size=32,
                    validation_data=(X_val_96, y_val),
                    callbacks=[callbacks.EarlyStopping(patience=4, restore_best_weights=True)],
                    verbose=0)
_, acc_ft = tl_model.evaluate(X_test_96, y_test, verbose=0)
print(f"After Fine-Tuning Accuracy: {acc_ft:.4f}  Δ={acc_ft-acc_tl:+.4f}")

# Comparison chart
labels  = ['Custom CNN', 'Augmented CNN', 'MobileNetV2 FE', 'MobileNetV2 FT']
accs    = [acc_orig, acc_aug, acc_tl, acc_ft]
colors  = ['#2196F3','#4CAF50','#FF9800','#E91E63']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, accs, color=colors, edgecolor='black', linewidth=0.8, width=0.55)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{acc:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0.5, 1.0); ax.set_ylabel('Test Accuracy')
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
# Section 8 — Hyperparameter Optimization

### 8.1 Learning Rate Sweep

In [ ]:
lrs = [1e-2, 1e-3, 5e-4, 1e-4]
lr_results = {}

for lr in lrs:
    m = build_world_cnn(name=f'lr_{lr}')
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train, epochs=10, batch_size=64,
              validation_data=(X_val, y_val), verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    lr_results[lr] = {'history': h, 'acc': acc}
    print(f"LR={lr:.0e}: test_acc={acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for lr, res in lr_results.items():
    axes[0].plot(res['history'].history['val_accuracy'], label=f'lr={lr:.0e}', lw=2)
    axes[1].plot(res['history'].history['val_loss'],     label=f'lr={lr:.0e}', lw=2)
for ax in axes: ax.legend(); ax.grid(True, alpha=0.3)
axes[0].set_title('Learning Rate — Val Accuracy')
axes[1].set_title('Learning Rate — Val Loss')
plt.suptitle('Learning Rate Sweep', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/lr_sweep.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.2 Batch Size Sweep

In [ ]:
batch_sizes = [32, 64, 128, 256]
bs_results  = {}

for bs in batch_sizes:
    m = build_world_cnn(name=f'bs_{bs}')
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train, epochs=10, batch_size=bs,
              validation_data=(X_val, y_val), verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    bs_results[bs] = {'history': h, 'acc': acc}
    print(f"batch_size={bs:>3}: test_acc={acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for bs, res in bs_results.items():
    ax.plot(res['history'].history['val_accuracy'], label=f'bs={bs}', lw=2)
ax.set_title('Batch Size Effect on Val Accuracy', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/plots/batch_size_sweep.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.3 Random Search — Best Hyperparameters

In [ ]:
param_space = {
    'lr'         : [1e-3, 5e-4, 1e-4],
    'batch_size' : [32, 64, 128],
    'filters'    : [32, 64],
    'dropout'    : [0.25, 0.4, 0.5],
}

all_combos = list(itertools.product(*param_space.values()))
sample     = random.sample(all_combos, min(8, len(all_combos)))

search_results = []
for i, (lr, bs, filt, drop) in enumerate(sample):
    def build_search(f=filt, d=drop):
        inp = keras.Input(shape=(32,32,3))
        x = layers.Conv2D(f,   (3,3), padding='same', activation='relu')(inp)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(d)(x)
        x = layers.Conv2D(f*2, (3,3), padding='same', activation='relu')(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(d)(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(d)(x)
        out = layers.Dense(N_CLASSES, activation='softmax')(x)
        return keras.Model(inp, out)
    m = build_search()
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(X_train, y_train, epochs=6, batch_size=bs,
          validation_data=(X_val, y_val), verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    search_results.append({'lr':lr,'batch':bs,'filters':filt,'dropout':drop,'acc':acc})
    print(f"Trial {i+1:>2}: lr={lr:.0e} bs={bs} f={filt} d={drop} → {acc:.4f}")

search_results.sort(key=lambda x: -x['acc'])

print("\n" + "═"*60)
print("RANDOM SEARCH — TOP RESULTS")
print("═"*60)
print(f"{'Rank':>4} {'LR':>8} {'Batch':>6} {'Filters':>8} {'Dropout':>8} {'Acc':>8}")
print("─"*60)
for rank, r in enumerate(search_results, 1):
    print(f"{rank:>4} {r['lr']:>8.0e} {r['batch']:>6} {r['filters']:>8} {r['dropout']:>8.2f} {r['acc']:>8.4f}")

best = search_results[0]
print(f"\n✅ Best config: LR={best['lr']:.0e}, Batch={best['batch']}, Filters={best['filters']}, Dropout={best['dropout']:.2f}")
print(f"   Best accuracy: {best['acc']:.4f}")


---
# Section 9 — Model Deployment

### 9.1 Save the Trained Model

In [ ]:
# Save in multiple formats
# 1. Keras native format
model.save('outputs/models/worldcnn_final.keras')
print("Saved: outputs/models/worldcnn_final.keras")

# 2. SavedModel format (TensorFlow serving)
model.export('outputs/models/worldcnn_savedmodel')
print("Saved: outputs/models/worldcnn_savedmodel/")

# 3. Weights only
model.save_weights('outputs/models/worldcnn_weights.weights.h5')
print("Saved: outputs/models/worldcnn_weights.weights.h5")

import os
for root, dirs, files in os.walk('outputs/models'):
    for f in files:
        fpath = os.path.join(root, f)
        size  = os.path.getsize(fpath) / 1024
        print(f"  {fpath}  ({size:.1f} KB)")


### 9.2 Load Model & Verify

In [ ]:
# Loading procedure documentation
print("═"*55)
print("MODEL LOADING PROCEDURE")
print("═"*55)
print("""
# Method 1 — Load full Keras model
loaded_model = keras.models.load_model('outputs/models/worldcnn_final.keras')

# Method 2 — Load from SavedModel directory
loaded_model = tf.saved_model.load('outputs/models/worldcnn_savedmodel')

# Method 3 — Rebuild architecture and load weights only
model = build_world_cnn()
model.load_weights('outputs/models/worldcnn_weights.weights.h5')
""")

# Verify loaded model gives identical results
loaded = keras.models.load_model('outputs/models/worldcnn_final.keras')
_, loaded_acc = loaded.evaluate(X_test, y_test, verbose=0)
_, orig_acc   = model.evaluate(X_test, y_test, verbose=0)
print(f"Original model accuracy : {orig_acc:.6f}")
print(f"Loaded model accuracy   : {loaded_acc:.6f}")
print(f"Match: {'✅ YES' if abs(loaded_acc - orig_acc) < 1e-5 else '❌ NO'}")


### 9.3 Interactive Image Prediction Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# ─── Prediction engine ───────────────────────────────────
def predict_image(img_array_uint8):
    """Predict class for a uint8 RGB numpy array (any size)."""
    img_resized = cv2.resize(img_array_uint8, (32, 32))
    img_norm    = img_resized.astype('float32') / 255.0
    img_batch   = np.expand_dims(img_norm, axis=0)
    probs       = loaded.predict(img_batch, verbose=0)[0]
    pred_idx    = np.argmax(probs)
    return pred_idx, probs

def show_prediction_chart(probs):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors  = ['#FF5722' if i == np.argmax(probs) else '#90CAF9' for i in range(N_CLASSES)]
    bars    = ax1.barh(CLASS_NAMES, probs, color=colors, edgecolor='black', lw=0.5)
    for bar, p in zip(bars, probs):
        ax1.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                 f'{p:.1%}', va='center', fontsize=9)
    ax1.set_xlim(0, 1.15)
    ax1.set_title('Prediction Probabilities', fontweight='bold')
    ax1.set_xlabel('Confidence'); ax1.axvline(0.5, color='red', ls='--', alpha=0.4)

    # Top-3 pie
    top3_idx  = np.argsort(probs)[-3:][::-1]
    top3_lbl  = [CLASS_NAMES[i] for i in top3_idx]
    top3_prob = probs[top3_idx]
    ax2.pie(top3_prob, labels=[f'{l}\n{p:.1%}' for l,p in zip(top3_lbl, top3_prob)],
            colors=['#FF5722','#FFC107','#2196F3'], startangle=90, wedgeprops={'edgecolor':'white','lw':1.5})
    ax2.set_title('Top-3 Predictions', fontweight='bold')

    plt.suptitle(f'Prediction: {CLASS_NAMES[np.argmax(probs)].upper()}  ({probs.max():.1%} confidence)',
                 fontsize=13, fontweight='bold', color='#FF5722')
    plt.tight_layout()
    plt.savefig('outputs/plots/latest_prediction.png', dpi=130, bbox_inches='tight')
    plt.show()

# ─── Demo on 6 random test images ────────────────────────
print("Demo: predicting 6 random test images")
print("─"*50)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.patch.set_facecolor('#1a1a2e')

sample_idx = np.random.choice(len(X_test), 6, replace=False)
for ax, idx in zip(axes.flat, sample_idx):
    pred_cls, probs = predict_image((X_test[idx]*255).astype(np.uint8))
    true_cls = np.argmax(y_test[idx])
    correct  = (pred_cls == true_cls)
    conf     = probs[pred_cls]

    ax.imshow(X_test[idx]); ax.axis('off')
    icon   = '✓' if correct else '✗'
    color  = '#00e676' if correct else '#ff1744'
    ax.set_title(f"{icon} True: {CLASS_NAMES[true_cls]}\n"
                 f"Pred: {CLASS_NAMES[pred_cls]} ({conf:.0%})",
                 color=color, fontsize=9.5, fontweight='bold')
    for sp in ax.spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(2.5)

plt.suptitle('Interactive Prediction Demo', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/prediction_demo.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

# ─── Widget Interface ─────────────────────────────────────
print("\n" + "═"*55)
print("INTERACTIVE PREDICTION WIDGET")
print("═"*55)

out   = widgets.Output()
btn   = widgets.Button(description='🎲 Predict Random Image', button_style='info',
                        layout=widgets.Layout(width='220px'))
label = widgets.Label(value='Click the button to predict a random CIFAR-10 test image.')

def on_button_click(b):
    with out:
        clear_output(wait=True)
        idx = np.random.randint(len(X_test))
        img_uint8 = (X_test[idx] * 255).astype(np.uint8)
        pred_cls, probs = predict_image(img_uint8)
        true_cls = np.argmax(y_test[idx])
        correct  = pred_cls == true_cls
        print(f"  True label : {CLASS_NAMES[true_cls]}")
        print(f"  Predicted  : {CLASS_NAMES[pred_cls]}  ({probs[pred_cls]:.1%}) {'✅' if correct else '❌'}")
        show_prediction_chart(probs)

btn.on_click(on_button_click)
display(widgets.VBox([label, btn, out]))


---
# 📊 Final Project Summary

In [ ]:
print("╔══════════════════════════════════════════════════════╗")
print("║         WORLD CNN SYSTEM — FINAL RESULTS            ║")
print("╠══════════════════════════════════════════════════════╣")
models_summary = {
    'Custom WorldCNN (baseline)' : acc_orig,
    'Deeper CNN'                  : acc_deeper,
    'WorldCNN + Augmentation'     : acc_aug,
    'MobileNetV2 (feature extr.)' : acc_tl,
    'MobileNetV2 (fine-tuned)'    : acc_ft,
}
for name, acc in models_summary.items():
    bar = '█' * int(acc * 40)
    print(f"║  {name:<28} {acc:.4f}  {bar}")
print("╚══════════════════════════════════════════════════════╝")

best_model = max(models_summary, key=models_summary.get)
best_acc   = models_summary[best_model]
print(f"\n🏆 Best model : {best_model}")
print(f"   Accuracy   : {best_acc:.4f}  ({best_acc*100:.2f}%)")
print("\n📁 Saved artefacts:")
for root, dirs, files in os.walk('outputs'):
    for f in files:
        print(f"  outputs/{os.path.relpath(os.path.join(root,f),'outputs')}")
